# 🧬 Project 5 — Semantic Correspondence (OFFICIAL FINAL V2)
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data
# Scaricamento PF-Pascal (Mirror Willow)
!mkdir -p ./data/PF-Pascal && wget -qO ./data/PF-Pascal/pfpascal.zip https://www.di.ens.fr/willow/research/proposal/dataset/PF-Pascal.zip && unzip -qo ./data/PF-Pascal/pfpascal.zip -d ./data/PF-Pascal/

import os, torch, gc
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print('[INFO] GPU Cleared.')

## 🔍 1. Baseline Evaluation

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt

## 🚀 2. Training (LoRA vs BitFit)

In [ ]:
# 2.1 Training LoRA
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
if not os.path.exists(CKPT_LORA):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir "$DRIVE_CKPTS/lora_only"
else: print('[OK] LoRA checkpoint trovato.')

In [ ]:
# 2.2 Training BitFit
CKPT_BITFIT = f'{DRIVE_CKPTS}/bitfit_only/bitfit_only_best.pth'
if not os.path.exists(CKPT_BITFIT):
    !python train.py --peft_type bitfit --dataset_root ./data/SPair-71k --epochs 5 --exp_name bitfit_only --output_dir ./checkpoints/bitfit_only --backup_dir "$DRIVE_CKPTS/bitfit_only"
else: print('[OK] BitFit checkpoint trovato.')

## 🎯 3. Raffinamento & Ablazione (LoRA vs BitFit)

In [ ]:
print('--- LoRA + AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/lora_aw.txt
print('--- BitFit + AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/bitfit_aw.txt

## 🌍 4. Generalizzazione & Robustezza (PF-Pascal)

In [ ]:
print('--- PF-Pascal: LoRA ---')
!python evaluate.py --dataset_root ./data/PF-Pascal/PF-Pascal --dataset_type pfpascal --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/gen_pascal_lora.txt
print('--- PF-Pascal: BitFit ---')
!python evaluate.py --dataset_root ./data/PF-Pascal/PF-Pascal --dataset_type pfpascal --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/gen_pascal_bitfit.txt

## ⚖️ 5. Demo Showcases

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

In [ ]:
launch_robustness_demo(ckpt_name='lora_only')